In [1]:
import torch
import numpy as np
import pandas as pd

from models.stardist import stardist_decode as stardist_decode_fn
import ioumatch

# Ton LightningModule
from models.lit_stardist_pannuke import StarDistLightning

device = "cuda" if torch.cuda.is_available() else "cpu"

ckpt_path = "models/checkpoints/stardist_pannuke_best.ckpt"
lit = StarDistLightning.load_from_checkpoint(ckpt_path)
lit = lit.to(device).eval()


In [3]:
from src.pannuke_dataset import PannukePreparedDataset
from pathlib import Path

DATA_ROOT = Path("data/prepared/pannuke")

val_raw = PannukePreparedDataset(
    root=DATA_ROOT,
    split="val",
    transform=None,             # comme ton notebook UNet
    return_image_name=True,
    load_dists=True,            # pas obligatoire pour eval pred complet, mais pratique
    n_rays=64,
    return_dists=True,          # si tu l’as dans ton dataset
)


In [4]:
def eval_stardist_on_val(
    val_ds,
    n=50,
    seed=0,
    prob_thr=0.5,
    nms_iou_thr=0.3,
    min_area=10,
    max_candidates=5000,
    vote_thr=0.5,
    iou_thr=0.5,
    use_local_maxima=False,
    local_max_footprint=9,
):
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(val_ds), size=min(n, len(val_ds)), replace=False)

    rows = []
    for idx in idxs:
        # selon ton dataset: (img, mask, types, dist, name)
        out = val_ds[idx]
        if len(out) == 5:
            img_t, gt_t, types_t, dist_t, name = out
        else:
            img_t, gt_t, types_t, name = out
            dist_t = None

        gt = gt_t.numpy().astype(np.int32)
        gt_bin = (gt > 0).astype(np.uint8)

        img_b = img_t.unsqueeze(0).to(device)

        with torch.no_grad():
            prob_logits, dist_pos, class_logits = lit.model(img_b)

        prob_map = torch.sigmoid(prob_logits)[0, 0].detach().cpu().numpy().astype(np.float32)
        dist_map = dist_pos[0].detach().cpu().numpy().astype(np.float32)          # (R,H,W), déjà en pixels chez toi
        class_prob = torch.softmax(class_logits, dim=1)[0].detach().cpu().numpy().astype(np.float32)

        pred_inst, pred_cls = stardist_decode_fn(
            prob_map=prob_map,
            dist_map=dist_map,
            class_prob=class_prob,
            prob_thr=float(prob_thr),
            nms_iou_thr=float(nms_iou_thr),
            use_local_maxima=bool(use_local_maxima),
            local_max_footprint=int(local_max_footprint),
            max_candidates=int(max_candidates),
            min_area=int(min_area),
            vote_thr=float(vote_thr),
        )

        pred_inst = pred_inst.astype(np.int32)

        # Pixel metrics
        pred_bin = (pred_inst > 0).astype(np.uint8)

        # Instance F1 via ioumatch (comme toi)
        res = ioumatch.evaluate_image(
            pred_inst,
            gt.astype(np.int32),
            threshold=float(iou_thr),
            method="greedy",
            inclusive=False,
            normalize=False,
        )

        rows.append({
            "image": name,
            "dice": dice_bin(pred_bin, gt_bin),
            "iou":  iou_bin(pred_bin, gt_bin),
            "f1":   float(res["f1"]),
            "tp":   int(res["tp"]),
            "fp":   int(res["fp"]),
            "fn":   int(res["fn"]),
            "gt_k": int(gt.max()),
            "pred_k": int(pred_inst.max()),
        })

    return pd.DataFrame(rows)


In [5]:
# Métriques (pixel + instance)

def dice_bin(pred_bin: np.ndarray, gt_bin: np.ndarray, eps=1e-7) -> float:
    pred = pred_bin.astype(bool)
    gt = gt_bin.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return float((2.0 * inter + eps) / (denom + eps))

def iou_bin(pred_bin: np.ndarray, gt_bin: np.ndarray, eps=1e-7) -> float:
    pred = pred_bin.astype(bool)
    gt = gt_bin.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return float((inter + eps) / (union + eps))

def f1_instance(pred_inst: np.ndarray, gt_inst: np.ndarray, iou_thr=0.5):
    res = ioumatch.evaluate_image(
        pred_inst.astype(np.int32),
        gt_inst.astype(np.int32),
        threshold=iou_thr,
        method="greedy",
        inclusive=False,
        normalize=False,
    )
    tp = int(res["tp"]); fp = int(res["fp"]); fn = int(res["fn"])
    f1 = (2*tp) / max((2*tp + fp + fn), 1e-9)
    return float(f1), tp, fp, fn
